# EX_07 — Reranking y optimización (ejercicios)

**Notebook de referencia:** `notebook/07_Reranking_Optimizacion.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Reordenar por cross-score simulado

Dada una query y 5 documentos, supón que tienes scores de un bi-encoder (baratos) y scores de un cross-encoder (caros). Implementa: tomar top-4 por bi-encoder y reordenar solo esos 4 por cross-score.


In [ ]:
import numpy as np

query = "latency vs throughput"
docs = [
    "doc0: latency is the time for one request",
    "doc1: throughput measures requests per second",
    "doc2: unrelated deployment notes",
    "doc3: latency and throughput trade-offs",
    "doc4: monitoring dashboard colors",
]
bi_scores = np.array([0.72, 0.81, 0.55, 0.78, 0.60])
cross_scores = np.array([0.10, 0.90, 0.20, 0.85, 0.30])

first_k = 4
candidate_indices = bi_scores.argsort()[::-1][:first_k]
reranked_indices = sorted(candidate_indices, key=lambda i: cross_scores[i], reverse=True)

print("Top candidates by bi-encoder:")
for i in candidate_indices:
    print(f"idx={i} bi={bi_scores[i]:.2f} cross={cross_scores[i]:.2f} | {docs[i]}")

print()
print("Final order after cross-encoder rerank:")
for rank, i in enumerate(reranked_indices, 1):
    print(f"{rank}. idx={i} cross={cross_scores[i]:.2f} | {docs[i]}")

## Actividad 2 — MMR esquemático

En pseudocódigo en Python (sin librería), bosqueja 5 líneas de selección **MMR** (balance relevancia / diversidad).


In [ ]:
def mmr_select(doc_ids, relevance, similarity, lambda_mult=0.7, top_k=3):
    """Minimal MMR selection: balance relevance to query and diversity vs selected docs."""
    selected = []
    candidates = list(doc_ids)

    while candidates and len(selected) < top_k:
        best_doc = None
        best_score = float("-inf")
        for doc_id in candidates:
            diversity_penalty = 0 if not selected else max(similarity[(doc_id, s)] for s in selected)
            mmr_score = lambda_mult * relevance[doc_id] - (1 - lambda_mult) * diversity_penalty
            if mmr_score > best_score:
                best_doc = doc_id
                best_score = mmr_score
        selected.append(best_doc)
        candidates.remove(best_doc)

    return selected

relevance = {"A": 0.95, "B": 0.90, "C": 0.75, "D": 0.70}
similarity = {
    ("A", "A"): 1.0, ("A", "B"): 0.85, ("A", "C"): 0.20, ("A", "D"): 0.10,
    ("B", "A"): 0.85, ("B", "B"): 1.0, ("B", "C"): 0.25, ("B", "D"): 0.15,
    ("C", "A"): 0.20, ("C", "B"): 0.25, ("C", "C"): 1.0, ("C", "D"): 0.40,
    ("D", "A"): 0.10, ("D", "B"): 0.15, ("D", "C"): 0.40, ("D", "D"): 1.0,
}
print(mmr_select(["A", "B", "C", "D"], relevance, similarity))

## Actividad 3 — Latencia

Estima en markdown (tabla breve) coste relativo: embedding único de query, k llamadas cross-encoder, generación LLM 200 tokens.


| Etapa | Coste relativo | Comentario |
|--------|----------------|------------|
| Embedding unico de query | 1x | Una sola pasada por el bi-encoder; barato y cacheable si hay queries repetidas. |
| Cross-encoder para k=20 | 20x-60x | Evalua pares query-documento; mas caro pero mejora precision del top final. |
| Generacion LLM ~200 tokens | 50x-200x | Suele dominar coste/latencia si el modelo es grande o remoto. |

Mitigacion: recuperar muchos candidatos con bi-encoder, reordenar solo top-k moderado con cross-encoder y enviar al LLM un contexto pequeno y bien citado.